In [ ]:
import requests
import pandas as pd
from IPython.display import display

def fetch_nea_forecast():
    print("Fetching NEA 2-Hour Weather Forecast...")
    url = "https://api.data.gov.sg/v1/environment/2-hour-weather-forecast"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        # The actual forecasts are buried inside the 'items' list
        forecasts = data['items'][0]['forecasts']
        df = pd.DataFrame(forecasts)

        print(f"✅ Success! Found {len(df)} area forecasts.")
        display(df.head()) # display() renders nice tables in Colab
        return df
    else:
        print(f"❌ Failed with status code: {response.status_code}")

def fetch_nea_temperature_readings():
    """
    This uses the Air Temperature API as a structural proxy.
    The WBGT and Lightning APIs will have this exact same nested JSON shape
    (combining station metadata with the actual reading values).
    """
    print("\nFetching NEA Station Readings...")
    url = "https://api.data.gov.sg/v1/environment/air-temperature"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        # 1. Extract the station metadata (Lat, Long, Name)
        stations = data['metadata']['stations']
        df_stations = pd.json_normalize(stations)

        # 2. Extract the actual temperature readings
        readings = data['items'][0]['readings']
        df_readings = pd.DataFrame(readings)

        # 3. Merge them together on the 'id' / 'station_id' column
        df_merged = pd.merge(
            df_readings,
            df_stations,
            left_on='station_id',
            right_on='id'
        ).drop(columns=['id'])

        print(f"✅ Success! Mapped {len(df_merged)} station readings.")
        display(df_merged.head())
        return df_merged
    else:
        print(f"❌ Failed with status code: {response.status_code}")

def fetch_onemap_route():
    print("\nFetching OneMap Walking Route...")
    # Example coordinates: Bishan Park area
    start_coords = "1.3600,103.8390"
    end_coords = "1.3500,103.8300"

    url = f"https://www.onemap.gov.sg/api/routing/v2/route?start={start_coords}&end={end_coords}&routeType=walk"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        # OneMap returns a set of turn-by-turn instructions
        instructions = data.get('route_instructions', [])
        df = pd.DataFrame(instructions)

        # We also want to grab the summary metrics
        distance = data['route_summary']['total_distance']
        time = data['route_summary']['total_time']

        print(f"✅ Success! Total Distance: {distance}m, Est Time: {time}s")
        display(df[['instruction', 'distance', 'time']].head())
        return df
    else:
        print(f"❌ Failed with status code: {response.status_code}")
        print(response.text)

# --- RUN THE FUNCTIONS ---
forecast_df = fetch_nea_forecast()
temperature_df = fetch_nea_temperature_readings()
route_df = fetch_onemap_route()

Fetching NEA 2-Hour Weather Forecast...
✅ Success! Found 47 area forecasts.


,area,forecast
0,Ang Mo Kio,Passing Showers
1,Bedok,Passing Showers
2,Bishan,Passing Showers
3,Boon Lay,Passing Showers
4,Bukit Batok,Passing Showers



Fetching NEA Station Readings...
✅ Success! Mapped 3 station readings.


,station_id,value,device_id,name,location.latitude,location.longitude
0,S107,25,S107,East Coast Parkway,1.31350,103.96250
1,S24,25,S24,Upper Changi Road North,1.36780,103.98260
2,S104,24,S104,Woodlands Avenue 9,1.44387,103.78538



Fetching OneMap Walking Route...
❌ Failed with status code: 403
{"message":"Missing Authentication Token"}


In [ ]:
import requests
import pandas as pd
from IPython.display import display

# --- 1. NEW: OneMap Authentication ---
def get_onemap_token(email, password):
    print("Authenticating with OneMap...")
    url = "https://www.onemap.gov.sg/api/auth/post/getToken"
    payload = {
        "email": email,
        "password": password
    }

    response = requests.post(url, json=payload)
    if response.status_code == 200:
        print("✅ Token successfully generated!")
        return response.json().get('access_token')
    else:
        print(f"❌ Authentication Failed: {response.status_code}")
        print(response.text)
        return None

# --- 2. NEA APIs (Still Public/No Token Needed) ---
def fetch_nea_forecast():
    print("\nFetching NEA 2-Hour Weather Forecast...")
    url = "https://api.data.gov.sg/v1/environment/2-hour-weather-forecast"
    response = requests.get(url)
    if response.status_code == 200:
        forecasts = response.json()['items'][0]['forecasts']
        df = pd.DataFrame(forecasts)
        print(f"✅ Success! Found {len(df)} area forecasts.")
        display(df.head())
        return df
    else:
        print(f"❌ Failed: {response.status_code}")

def fetch_nea_temperature_readings():
    print("\nFetching NEA Station Readings...")
    url = "https://api.data.gov.sg/v1/environment/air-temperature"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        df_stations = pd.json_normalize(data['metadata']['stations'])
        df_readings = pd.DataFrame(data['items'][0]['readings'])

        df_merged = pd.merge(df_readings, df_stations, left_on='station_id', right_on='id').drop(columns=['id'])
        print(f"✅ Success! Mapped {len(df_merged)} station readings.")
        display(df_merged.head())
        return df_merged
    else:
        print(f"❌ Failed: {response.status_code}")

# --- 3. UPDATED: OneMap Routing with Correct URL ---
def fetch_onemap_route(token):
    if not token:
        print("\n❌ Skipping Route fetch: No token provided.")
        return

    print("\nFetching OneMap Walking Route...")
    start_coords = "1.3600,103.8390"
    end_coords = "1.3500,103.8300"

    # FIX 1: The brand new routing endpoint URL
    url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={start_coords}&end={end_coords}&routeType=walk"

    # FIX 2: "Bearer " is required for the new public endpoints
    headers = {
        "Authorization": f"Bearer {token}"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
            data = response.json()

            # Grab the raw instructions
            raw_instructions = data.get('route_instructions', [])
            df = pd.DataFrame(raw_instructions)

            distance = data.get('route_summary', {}).get('total_distance', 'Unknown')
            print(f"✅ Success! Total Distance: {distance}m")

            # FIX: Just print the raw DataFrame exactly as it comes in!
            display(df.head())
            return df
    else:
        print(f"❌ Failed: {response.status_code}")
        print(response.text)

# ==========================================
# RUN THE SCRIPT HERE
# ==========================================

# 1. Put your registered OneMap email and password here
ONEMAP_EMAIL = "put your registration email"
ONEMAP_PASSWORD = "account password"

# 2. Get the token
my_token = get_onemap_token(ONEMAP_EMAIL, ONEMAP_PASSWORD)

# 3. Fetch the data
forecast_df = fetch_nea_forecast()
temperature_df = fetch_nea_temperature_readings()
route_df = fetch_onemap_route(my_token)

Authenticating with OneMap...
✅ Token successfully generated!

Fetching NEA 2-Hour Weather Forecast...
✅ Success! Found 47 area forecasts.


,area,forecast
0,Ang Mo Kio,Passing Showers
1,Bedok,Passing Showers
2,Bishan,Passing Showers
3,Boon Lay,Passing Showers
4,Bukit Batok,Passing Showers



Fetching NEA Station Readings...
✅ Success! Mapped 2 station readings.


,station_id,value,device_id,name,location.latitude,location.longitude
0,S107,26.3,S107,East Coast Parkway,1.3135,103.9625
1,S24,28.5,S24,Upper Changi Road North,1.3678,103.9826



Fetching OneMap Walking Route...
✅ Success! Total Distance: 3259m


,0,1,2,3,4,5,6,7,8,9
0,Right,,181,"1.360163,103.839081",130,181m,South East,North,walking,Head Southeast
1,Right,,100,"1.359142,103.839649",72,100m,West,South West,walking,Turn Right
2,Left,,152,"1.358579,103.839097",110,152m,South East,South West,walking,Turn Left
3,Straight,,270,"1.35747,103.8386",194,270m,West,South West,walking,Continue Straight
4,Left,,4,"1.355598,103.83725",3,4m,South,West,walking,Turn Left


In [ ]:
import requests
import pandas as pd
from IPython.display import display

def fetch_v2_lightning_data():
    print("Fetching v2 Lightning Data...")

    # The new v2 open API endpoint for lightning
    url = "https://api-open.data.gov.sg/v2/real-time/api/weather?api=lightning"

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print("✅ Success! Connected to v2 Lightning API.")

        # Because this is a new endpoint, let's print the top-level keys
        # to understand the exact JSON architecture.
        print("\n--- Top Level JSON Keys ---")
        print(list(data.keys()))

        # Safely try to extract the main payload
        # v2 APIs usually nest the core payload inside 'data', 'items', or 'records'
        if 'data' in data:
            target_list = data['data']
        elif 'items' in data:
            target_list = data['items']
        else:
            target_list = data

        print("\n--- Creating DataFrame ---")
        # Attempt to flatten it into a Pandas table
        try:
            # df = pd.json_normalize(target_list)
            # print(f"Total Records found: {len(df)}")
            # display(df.head())
            # return df
            # Dig straight into the items array
            strikes = data['data']['records'][0]['items']
            display(pd.DataFrame(strikes))
        except Exception as e:
            print("Could not automatically flatten. Here is the raw data:")
            display(data)
            return data

    else:
        print(f"❌ Failed: {response.status_code}")
        print(response.text)

# Run the function
lightning_df = fetch_v2_lightning_data()



Fetching v2 Lightning Data...
✅ Success! Connected to v2 Lightning API.

--- Top Level JSON Keys ---
['code', 'data', 'errorMsg']

--- Creating DataFrame ---
Could not automatically flatten. Here is the raw data:


{'code': 0,
 'data': {'records': [{'datetime': '2026-03-15T14:52:00+08:00',
    'item': {'isStationData': False, 'readings': [], 'type': 'observation'},
    'updatedTimestamp': '2026-03-15T14:54:02+08:00'}]},
 'errorMsg': ''}

In [ ]:
import requests
import pandas as pd
from IPython.display import display

def fetch_v2_lightning_data():
    print("Fetching v2 Lightning Data...")

    url = "https://api-open.data.gov.sg/v2/real-time/api/weather?api=lightning"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print("✅ Success! Connected to v2 Lightning API.\n")

        try:
            # The exact path based on your live test!
            strikes = data['data']['records'][0]['item']['readings']

            if len(strikes) == 0:
                print("🌤️ Good news! The 'readings' array is empty.")
                print("There are currently 0 lightning strikes in Singapore right now.")
                return pd.DataFrame() # Return an empty table safely
            else:
                df = pd.DataFrame(strikes)
                print(f"⚡ WARNING: {len(df)} Lightning Strikes detected!")
                display(df.head())
                return df

        except KeyError as e:
            print(f"❌ Could not find the expected keys. The API structure might have changed. Missing: {e}")
            display(data)
            return None

    else:
        print(f"❌ Failed: {response.status_code}")
        print(response.text)

# Run the function
lightning_df = fetch_v2_lightning_data()

Fetching v2 Lightning Data...
✅ Success! Connected to v2 Lightning API.

⚡ WARNING: 32 Lightning Strikes detected!


,location,type,text,datetime
0,"{'latitude': '1.5385', 'longitude': '103.4139'}",C,Cloud to Cloud,2026-03-15T14:52:10.534+08:00
1,"{'latitude': '1.5769', 'longitude': '103.3355'}",C,Cloud to Cloud,2026-03-15T14:52:10.547+08:00
2,"{'latitude': '1.5452', 'longitude': '103.3947'}",C,Cloud to Cloud,2026-03-15T14:52:10.553+08:00
3,"{'latitude': '1.5452', 'longitude': '103.3947'}",C,Cloud to Cloud,2026-03-15T14:52:10.553+08:00
4,"{'latitude': '1.5193', 'longitude': '103.4540'}",C,Cloud to Cloud,2026-03-15T14:52:10.733+08:00


In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import date, timedelta

BASE = "https://api-open.data.gov.sg/v2/real-time/api"

ENDPOINTS = [
    ("air-temperature", "air_temperature"),
    ("rainfall", "rainfall"),
    ("relative-humidity", "relative_humidity"),
    ("wind-direction", "wind_direction"),
    ("wind-speed", "wind_speed"),
]

API_KEY = None  # paste your API key here if you have one (recommended)
HEADERS = {"Accept": "application/json"}
if API_KEY:
    HEADERS["x-api-key"] = API_KEY

session = requests.Session()

def get_json_with_backoff(url, max_retries=12):
    wait = 2
    for attempt in range(1, max_retries + 1):
        r = session.get(url, headers=HEADERS, timeout=30)

        if r.status_code == 429:
            sleep_for = max(10, wait)
            print(f"[429] Rate limited. Sleep {sleep_for}s (attempt {attempt}/{max_retries})")
            time.sleep(sleep_for)
            wait = min(wait * 2, 60)
            continue

        if r.status_code != 200:
            raise RuntimeError(f"{r.status_code} {r.text[:200]}")

        return r.json()

    raise RuntimeError("Too many retries (429). Add API key or slow down more.")

def fetch_full_day_long(endpoint: str, metric: str, day_str: str, base_sleep=0.6, max_pages=500):
    token = None
    pages = 0
    stations = None
    readings_all = []

    while True:
        url = f"{BASE}/{endpoint}?date={day_str}"
        if token:
            url += f"&paginationToken={token}"

        j = get_json_with_backoff(url)
        data = j.get("data", {}) or {}

        if stations is None:
            stations = data.get("stations", []) or []

        readings_all.extend(data.get("readings", []) or [])
        token = data.get("paginationToken")

        pages += 1
        time.sleep(base_sleep)

        if not token:
            break
        if pages >= max_pages:
            raise RuntimeError(f"Too many pages for {endpoint} {day_str}")

    # station lookup
    st_df = pd.DataFrame([{
        "stationId": s.get("id"),
        "stationName": s.get("name"),
        "latitude": (s.get("labelLocation") or {}).get("latitude"),
        "longitude": (s.get("labelLocation") or {}).get("longitude"),
    } for s in stations])

    # flatten readings
    rows = []
    for r in readings_all:
        ts = r.get("timestamp")
        for dpt in r.get("data", []):
            rows.append({
                "timestamp": ts,
                "stationId": dpt.get("stationId"),
                "metric": metric,
                "value": dpt.get("value"),   # keep raw; convert after pivot
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df, st_df, pages

    df = df.merge(st_df, on="stationId", how="left")
    return df, st_df, pages

# ---- choose ONE day ----
DAY = "2026-03-01"
# DAY = (date.today() - timedelta(days=1)).isoformat()  # yesterday

all_long = []
station_master = None

for endpoint, metric in ENDPOINTS:
    print(f"\n=== {endpoint} ({DAY}) ===")
    df_long, st_df, pages = fetch_full_day_long(endpoint, metric, DAY, base_sleep=0.6)
    print("pages:", pages, "rows:", len(df_long))
    all_long.append(df_long)

    # Build a station master (merge later for lat/lon)
    if station_master is None:
        station_master = st_df

    time.sleep(1.2)

long_df = pd.concat([d for d in all_long if not d.empty], ignore_index=True)
print("\nLONG shape:", long_df.shape)

# ---- pivot to WIDE (IMPORTANT: do NOT include lat/lon in the index) ----
wide_df = (
    long_df
    .pivot_table(
        index=["timestamp", "stationId", "stationName"],
        columns="metric",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Convert metric columns after pivot
metric_cols = ["air_temperature", "rainfall", "relative_humidity", "wind_direction", "wind_speed"]
for c in metric_cols:
    if c in wide_df.columns:
        wide_df[c] = pd.to_numeric(wide_df[c], errors="coerce")

# Merge lat/lon after pivot (won't drop rows even if lat/lon missing)
if station_master is not None and not station_master.empty:
    wide_df = wide_df.merge(
        station_master[["stationId", "latitude", "longitude"]],
        on="stationId",
        how="left"
    )

# Save to Desktop
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
outpath = os.path.join(desktop, "sg_weather_1day_wide.csv")
wide_df.to_csv(outpath, index=False)

print("\nSaved:", outpath)
print("WIDE rows:", len(wide_df))
print(wide_df.head())


In [ ]:
import requests
import pandas as pd
from datetime import date, timedelta
import time

BASE = "https://api-open.data.gov.sg/v2/real-time/api/weather"
OUTFILE = "wbgt_past_30_days.csv"

session = requests.Session()

def get_json(url, retries=3, sleep=2):
    for attempt in range(1, retries + 1):
        resp = session.get(url, timeout=30)
        if resp.status_code != 200:
            print(f"[{resp.status_code}] attempt {attempt}: {url}")
            print(resp.text[:300])  # show first 300 chars
            time.sleep(sleep)
            continue
        try:
            return resp.json()
        except Exception as e:
            print(f"[JSON parse error] attempt {attempt}: {e}")
            print(resp.text[:300])
            time.sleep(sleep)
    return None

def fetch_day(day_str: str):
    url = f"{BASE}?api=wbgt&date={day_str}"
    j = get_json(url)

    if not isinstance(j, dict):
        print(f"Skipping {day_str}: no JSON returned")
        return []

    data = j.get("data")
    if not isinstance(data, dict):
        print(f"Skipping {day_str}: missing 'data' field. code={j.get('code')} err={j.get('errorMsg')}")
        return []

    records = data.get("records", [])
    rows = []
    for r in records:
        dt = r.get("datetime")
        readings = r.get("item", {}).get("readings", [])
        for x in readings:
            rows.append({
                "datetime": dt,
                "station_id": x.get("station", {}).get("id"),
                "station_name": x.get("station", {}).get("name"),
                "townCenter": x.get("station", {}).get("townCenter"),
                "latitude": x.get("location", {}).get("latitude"),
                "longitude": x.get("location", {}).get("longitude"),
                "wbgt": x.get("wbgt"),
                "heatStress": x.get("heatStress"),
            })
    return rows

# Pick your window
end_day = date(2026, 3, 1)          # adjust
start_day = end_day - timedelta(days=29)

all_rows = []
d = start_day
while d <= end_day:
    day_str = d.isoformat()
    rows = fetch_day(day_str)
    print(day_str, "rows:", len(rows))
    all_rows.extend(rows)
    d += timedelta(days=1)

df = pd.DataFrame(all_rows)

if not df.empty:
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["wbgt"] = pd.to_numeric(df["wbgt"], errors="coerce")

df.to_csv(OUTFILE, index=False)
print("Saved:", OUTFILE, "rows:", len(df))


## Unified Ingestion → LLM Run Decision (MVP)

This section gives you a clean end-to-end prototype:

1. Pull live data from NEA APIs (forecast, temperature, lightning, WBGT)
2. Build a compact structured context object
3. Send context to an LLM API for a `RUN` / `MODIFY` / `DO_NOT_RUN` decision
4. Fall back to rule-based logic if LLM is unavailable

In [ ]:
import os
import json
import requests
import pandas as pd
from datetime import datetime
from IPython.display import display

# -----------------------------
# API clients (live ingestion)
# -----------------------------
def fetch_nea_2h_forecast():
    url = "https://api.data.gov.sg/v1/environment/2-hour-weather-forecast"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    payload = r.json()

    forecasts = payload.get("items", [{}])[0].get("forecasts", [])
    ts = payload.get("items", [{}])[0].get("timestamp")
    return {
        "timestamp": ts,
        "forecasts": forecasts,
        "forecast_count": len(forecasts),
    }


def fetch_nea_air_temperature():
    url = "https://api.data.gov.sg/v1/environment/air-temperature"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    payload = r.json()

    stations = pd.json_normalize(payload.get("metadata", {}).get("stations", []))
    readings = pd.DataFrame(payload.get("items", [{}])[0].get("readings", []))

    if readings.empty:
        return {"timestamp": None, "station_count": 0, "avg_temp_c": None, "max_temp_c": None}

    merged = readings.merge(stations, left_on="station_id", right_on="id", how="left")
    merged["value"] = pd.to_numeric(merged["value"], errors="coerce")

    ts = payload.get("items", [{}])[0].get("timestamp")
    return {
        "timestamp": ts,
        "station_count": int(merged.shape[0]),
        "avg_temp_c": float(merged["value"].mean(skipna=True)),
        "max_temp_c": float(merged["value"].max(skipna=True)),
    }


def fetch_v2_lightning():
    url = "https://api-open.data.gov.sg/v2/real-time/api/weather?api=lightning"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    payload = r.json()

    records = payload.get("data", {}).get("records", [])
    if not records:
        return {"timestamp": None, "strike_count": 0}

    first = records[0]
    readings = first.get("item", {}).get("readings", [])
    return {
        "timestamp": first.get("timestamp") or first.get("datetime"),
        "strike_count": len(readings),
    }


def fetch_v2_wbgt_latest():
    url = "https://api-open.data.gov.sg/v2/real-time/api/weather?api=wbgt"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    payload = r.json()

    records = payload.get("data", {}).get("records", [])
    if not records:
        return {"timestamp": None, "avg_wbgt": None, "max_wbgt": None, "heat_stress_levels": []}

    first = records[0]
    readings = first.get("item", {}).get("readings", [])
    if not readings:
        return {"timestamp": first.get("datetime"), "avg_wbgt": None, "max_wbgt": None, "heat_stress_levels": []}

    wbgt_vals = pd.to_numeric(pd.Series([x.get("wbgt") for x in readings]), errors="coerce")
    stress_levels = sorted({x.get("heatStress") for x in readings if x.get("heatStress")})

    return {
        "timestamp": first.get("datetime"),
        "avg_wbgt": float(wbgt_vals.mean(skipna=True)) if wbgt_vals.notna().any() else None,
        "max_wbgt": float(wbgt_vals.max(skipna=True)) if wbgt_vals.notna().any() else None,
        "heat_stress_levels": stress_levels,
    }


# -----------------------------
# Context builder
# -----------------------------
def build_run_context(target_area: str = "Bishan"):
    forecast = fetch_nea_2h_forecast()
    temp = fetch_nea_air_temperature()
    lightning = fetch_v2_lightning()
    wbgt = fetch_v2_wbgt_latest()

    area_forecast = None
    for row in forecast["forecasts"]:
        if str(row.get("area", "")).strip().lower() == target_area.strip().lower():
            area_forecast = row.get("forecast")
            break

    context = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "target_area": target_area,
        "nea_2h_forecast": {
            "timestamp": forecast["timestamp"],
            "area_forecast": area_forecast,
            "forecast_count": forecast["forecast_count"],
        },
        "air_temperature": temp,
        "lightning": lightning,
        "wbgt": wbgt,
    }
    return context


# -----------------------------
# Decision logic
# -----------------------------
def rule_based_decision(context: dict):
    max_wbgt = context.get("wbgt", {}).get("max_wbgt")
    strike_count = context.get("lightning", {}).get("strike_count", 0)
    area_fc = str(context.get("nea_2h_forecast", {}).get("area_forecast", "") or "").lower()

    reasons = []
    decision = "RUN"

    if strike_count > 0:
        decision = "DO_NOT_RUN"
        reasons.append("Lightning strikes detected in latest reading.")

    if max_wbgt is not None and max_wbgt >= 33:
        decision = "DO_NOT_RUN"
        reasons.append(f"WBGT is very high ({max_wbgt:.1f}).")
    elif max_wbgt is not None and max_wbgt >= 30 and decision != "DO_NOT_RUN":
        decision = "MODIFY"
        reasons.append(f"WBGT is elevated ({max_wbgt:.1f}); reduce intensity and duration.")

    rain_terms = ["thundery", "rain", "showers", "storm"]
    if any(t in area_fc for t in rain_terms) and decision == "RUN":
        decision = "MODIFY"
        reasons.append("Rain/thunder risk in 2-hour area forecast.")

    if not reasons:
        reasons.append("No major immediate weather risk detected.")

    return {
        "source": "rule_based",
        "decision": decision,
        "confidence": 0.65,
        "reasoning": reasons,
    }


def llm_decision(context: dict):
    """
    Generic LLM HTTP call. Configure via environment variables:
      - LLM_API_URL (required)
      - LLM_API_KEY (optional but usually required)
      - LLM_MODEL   (optional)

    Expected response format (JSON):
      {"decision":"RUN|MODIFY|DO_NOT_RUN", "confidence":0..1, "reasoning":[...], "notes":"..."}
    """
    api_url = os.getenv("LLM_API_URL")
    api_key = os.getenv("LLM_API_KEY")
    model = os.getenv("LLM_MODEL", "gpt-4o-mini")

    if not api_url:
        raise RuntimeError("LLM_API_URL is not set")

    system_prompt = (
        "You are a weather safety triage assistant for outdoor running in Singapore. "
        "Return strict JSON with keys: decision, confidence, reasoning, notes. "
        "decision must be one of RUN, MODIFY, DO_NOT_RUN."
    )

    user_payload = {
        "task": "Decide whether user should run now.",
        "policy": {
            "lightning_detected": "prefer DO_NOT_RUN",
            "wbgt_ge_33": "DO_NOT_RUN",
            "wbgt_30_to_32_9": "MODIFY"
        },
        "context": context,
    }

    # OpenAI-compatible Chat Completions style payload
    body = {
        "model": model,
        "temperature": 0.1,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(user_payload)},
        ],
    }

    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    r = requests.post(api_url, headers=headers, json=body, timeout=60)
    r.raise_for_status()

    out = r.json()
    content = out.get("choices", [{}])[0].get("message", {}).get("content", "{}")
    parsed = json.loads(content)

    return {
        "source": "llm",
        "decision": parsed.get("decision", "MODIFY"),
        "confidence": float(parsed.get("confidence", 0.5)),
        "reasoning": parsed.get("reasoning", []),
        "notes": parsed.get("notes", ""),
    }


def get_run_recommendation(target_area: str = "Bishan"):
    context = build_run_context(target_area=target_area)

    try:
        decision = llm_decision(context)
    except Exception as e:
        decision = rule_based_decision(context)
        decision["notes"] = f"LLM unavailable, fallback used: {e}"

    return {
        "context": context,
        "decision": decision,
    }

In [ ]:
# Optional: set env vars in-notebook for quick testing
# os.environ["LLM_API_URL"] = "https://your-llm-endpoint/v1/chat/completions"
# os.environ["LLM_API_KEY"] = "your_api_key"
# os.environ["LLM_MODEL"] = "gpt-4o-mini"

result = get_run_recommendation(target_area="Bishan")

print("Decision Source:", result["decision"]["source"])
print("Decision:", result["decision"]["decision"])
print("Confidence:", result["decision"]["confidence"])
print("Reasoning:")
for i, r in enumerate(result["decision"].get("reasoning", []), 1):
    print(f"  {i}. {r}")

print("\nContext Snapshot:")
display(pd.json_normalize(result["context"], sep="."))